# Fine-Tuning Supervisado de LLMs con Hugging Face (SFT y LoRA)

**Nivel:** Fundacional / Intermedio  
**Tecnologias:** Hugging Face `transformers`, `trl` (Transformer Reinforcement Learning), `peft` y `bitsandbytes`  
**Modelo Base:** Google Gemma 2 2B Instruct (`google/gemma-2-2b-it`) / Qwen 2.5 1.5B  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/01-sft-lora-huggingface/01_sft_lora_huggingface.ipynb)

---

## 1. Fundamentos Teoricos: Supervised Fine-Tuning (SFT) y PEFT

### El Rol del SFT en el Ciclo de Vida del LLM
Los modelos base (pre-entrenados) unicamente saben predecir el siguiente token sobre billones de paginas web. El **Supervised Fine-Tuning (SFT)** es la etapa donde el modelo aprende a:
1. Comprender la estructura de un dialogo conversacional (*turnos de usuario y asistente*).
2. Seguir instrucciones de forma disciplinada y respetar restricciones de formato.
3. Adoptar una personalidad, tono o conocimiento especializado en un dominio vertical (e.g. atencion al cliente, medicina, soporte tecnico, finanzas).

### Por que LoRA (Low-Rank Adaptation)?
Un fine-tuning completo (*Full Fine-Tuning*) requiere actualizar y almacenar todos los pesos de la red. Para un modelo de 2,000 millones de parametros, esto requiere mas de 16 GB de VRAM solo para los gradientes y estados del optimizador Adam.

**LoRA (Hu et al., 2021)** congela los pesos pre-entrenados $W_0 \in \mathbb{R}^{d \times k}$ y descompone la actualizacion en dos matrices de rango bajo:

$$\Delta W = B \times A$$

donde $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ y el rango $r \ll \min(d, k)$ (tipicamente $r=8$ o $r=16$).

- **Reduccion drastica de parametros:** Menos del 0.5% de los pesos totales son entrenables.
- **Cero latencia en inferencia:** Las matrices $B \times A$ se pueden fusionar matematicamente con $W_0$ (`merge_and_unload()`) para produccion.
- **Modularidad:** Un solo modelo base puede servir multiples adaptadores LoRA ligeros (~15-30 MB cada uno).

### Objetivos Pedagogicos de este Laboratorio
- Cargar un modelo de lenguaje abierto con tokenizador en formato `bfloat16` o cuantizacion 4-bit.
- Ejecutar una inferencia inicial (baseline) para evidenciar las limitaciones del modelo antes de ser ajustado.
- Estructurar un dataset de instrucciones y formatearlo con plantillas de chat estandarizadas (`apply_chat_template`).
- Configurar e inyectar adaptadores LoRA mediante la biblioteca `peft`.
- Entrenar el modelo con `SFTTrainer` de la biblioteca `trl`.
- Evaluar el cambio de comportamiento, tono y conocimiento en el modelo post-entrenamiento.
- Guardar los adaptadores LoRA y fusionarlos con los pesos base.


### Paso 1: Instalacion de Dependencias en Google Colab o Entorno Local

Instalamos el stack moderno de post-entrenamiento de Hugging Face:


### Configuración fuera de Google Colab

Si estás ejecutando este notebook en un entorno local, servidor propio o contenedor Docker, asegúrate de tener instaladas las siguientes bibliotecas base. A diferencia de Colab, estos entornos suelen estar vacíos:

*   **Motor de Deep Learning:** `torch` y `torchvision` (asegúrate de instalar la versión compatible con tu versión de CUDA).
*   **Ecosistema Hugging Face:** `transformers`, `accelerate` y `datasets`.
*   **Fine-Tuning y Optimización:** `peft` (para LoRA), `trl` (para el SFTTrainer) y `bitsandbytes` (para cuantización de 4/8 bits).

**Comando de instalación recomendado:**
```bash
pip install torch torchvision transformers accelerate datasets peft trl bitsandbytes
```

*Nota: Se recomienda el uso de entornos virtuales (`venv` o `conda`) para evitar conflictos de dependencias.*

In [1]:
# Instalamos solo lo que no viene por defecto en Colab y actualizamos lo mínimo necesario
!pip install -qU peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 29.8 MB/s eta 0:00:00


### Paso 2: Importacion de Bibliotecas y Verificacion de Hardware Acelerado

Verificamos la version de PyTorch y la disponibilidad de aceleracion GPU (NVIDIA CUDA o Apple MPS):


In [2]:
import torch
import transformers
import trl
import peft
import datasets
import os
import gc

print(f"PyTorch version:    {torch.__version__}")
print(f"Transformers:       {transformers.__version__}")
print(f"TRL version:        {trl.__version__}")
print(f"PEFT version:       {peft.__version__}")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Dispositivo activo: {device}")
if device == "cuda":
    print(f"GPU detectada:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM total:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version:    2.11.0+cu128
Transformers:       5.16.1
TRL version:        1.12.0
PEFT version:       0.20.0
Dispositivo activo: cuda
GPU detectada:      Tesla T4
VRAM total:         15.64 GB


### Paso 3: Seleccion y Carga del Modelo Base y Tokenizador

Utilizamos `google/gemma-2-2b-it` (o alternativamente `Qwen/Qwen2.5-1.5B-Instruct` como opcion ligera y de libre acceso). Configuramos precision `bfloat16` para estabilidad numerica en aceleradores modernos:


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-2-2b-it"

print(f"Cargando tokenizador de: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Garantizar que exista token de relleno (pad_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Opcional: Cuantizacion 4-bit para GPUs con memoria limitada (e.g. Colab T4 16GB)
use_4bit = torch.cuda.is_available()
quantization_config = None
if use_4bit:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

print(f"Cargando modelo base {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print("Modelo base cargado exitosamente.")

Cargando tokenizador de: google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Modelo base cargado exitosamente.


### Paso 4: Inferencia Inicial (Baseline Antes del Fine-Tuning)

Probamos una consulta especializada de atencion al cliente para un servicio corporativo ("TechCloud Pro"). El modelo base respondera de forma generica o admitira desconocer las politicas de la compania:


In [10]:
def test_prompt(user_query, model_instance, tokenizer_instance):
    # Forzamos que el modelo use el modo evaluación y limpie el cache de inferencia
    model_instance.eval()

    messages = [
        {"role": "user", "content": user_query}
    ]

    prompt = tokenizer_instance.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_instance(prompt, return_tensors="pt", add_special_tokens=False).to(model_instance.device)

    # Ajustamos parámetros de generación para evitar repeticiones
    with torch.no_grad():
        output_ids = model_instance.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.3, # Menor temperatura = más coherencia
            top_p=0.9,
            repetition_penalty=1.2, # Penaliza la repetición de los mismos tokens
            pad_token_id=tokenizer_instance.pad_token_id,
            eos_token_id=tokenizer_instance.eos_token_id
        )

    response_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer_instance.decode(response_tokens, skip_special_tokens=True).strip()

sample_query = "¿Qué tipo de soporte tiene el plan Enterprise y cuánto tardan en responder?"
print("=== RESPUESTA ANTES DEL FINE-TUNING (BASELINE) ===\n")
print(test_prompt(sample_query, model, tokenizer))

=== RESPUESTA ANTES DEL FINE-TUNING (BASELINE) ===

El TechCloud Enterprise ofrece un reembolso total en los primeros 48 horas luego de la compra. El proceso tarda de 3 a 5 dias habiles.


### Paso 5: Creacion del Dataset Instruccional de Dominio Especializado

Estructuramos un conjunto de datos en formato conversacional con preguntas y respuestas oficiales de "TechCloud Pro":


In [5]:
import random

# Base de conocimientos de TechCloud Pro
topics = [
    ("politica de reembolso", "El plan Enterprise ofrece reembolso total en los primeros 30 dias. El proceso tarda de 3 a 5 dias habiles."),
    ("rotar llaves de API", "En el portal, ve a Infraestructura > Clusters K8s > Seguridad > Rotar API Key. Las viejas expiran en 60 segundos."),
    ("cumplimiento HIPAA", "Si, cumplimos con HIPAA, SOC2 Tipo II e ISO 27001 con cifrado AES-256."),
    ("soporte tecnico", "Starter tiene foro (48h). Enterprise tiene TAM dedicado y respuesta en menos de 15 minutos 24/7."),
    ("ancho de banda", "Nodos con 25 Gbps internos y 10 Gbps publicos. Incluye 10 TB mensuales gratuitos."),
    ("bases de datos SQL", "TechCloud SQL soporta PostgreSQL y MySQL con replicacion automatica y backups cada 24 horas."),
    ("TechCloud Shield", "Es nuestro servicio de proteccion DDoS de capa 7 incluido por defecto en todos los planes Enterprise."),
    ("escalado Kubernetes", "Use el comando 'techcloud cluster scale --nodes=N' o active el Auto-scaling en el panel de control."),
    ("pago anual", "Ofrecemos un 20% de descuento en todos los planes si se realiza la contratacion por 12 meses por adelantado."),
    ("regiones", "TechCloud Pro opera en US-East, EU-West (Frankfurt) y Asia-Pacific (Tokyo).")
]

# Generar 100 variaciones sintéticas
raw_dataset = []
for _ in range(10): # 10 repeticiones con ligeras variaciones de parafraseo
    for q_base, a_base in topics:
        prefixes = ["¿Podrías decirme la", "Necesito saber sobre la", "Consulta sobre", "Información de", "Dime la"]
        prefix = random.choice(prefixes)
        raw_dataset.append({
            "messages": [
                {"role": "user", "content": f"{prefix} {q_base}?"},
                {"role": "assistant", "content": a_base}
            ]
        })

from datasets import Dataset
dataset = Dataset.from_list(raw_dataset)
print(f"Dataset expandido con éxito: {len(dataset)} muestras generadas.")
print("Ejemplo:", dataset[random.randint(0, 99)])

Dataset expandido con éxito: 100 muestras generadas.
Ejemplo: {'messages': [{'role': 'user', 'content': 'Necesito saber sobre la ancho de banda?'}, {'role': 'assistant', 'content': 'Nodos con 25 Gbps internos y 10 Gbps publicos. Incluye 10 TB mensuales gratuitos.'}]}


### Paso 6: Inyeccion de Adaptadores LoRA con PEFT

Configuramos `LoraConfig` definiendo el rango $r=8$, factor de escala $\alpha=16$ y apuntando a las matrices de proyeccion de atencion (`q_proj` y `v_proj`):


In [6]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


### Paso 7: Configuracion y Entrenamiento con `SFTTrainer` (Hugging Face TRL)

Utilizamos `SFTTrainer` de `trl`, el entrenador especializado de la industria para Supervised Fine-Tuning. `SFTConfig` gestiona el learning rate, acumulacion de gradientes y optimizador paginado AdamW:


In [7]:
from trl import SFTTrainer, SFTConfig
from peft import get_peft_model
import torch
import gc

# 1. Limpieza absoluta antes de empezar
if 'peft_model' in locals():
    del peft_model
    torch.cuda.empty_cache()
    gc.collect()

if hasattr(model, "unload"):
    model = model.unload()

# 2. Desactivamos el cache para el entrenamiento
model.config.use_cache = False

# 3. Configuración con Scheduler para evitar fluctuaciones agresivas
training_args = SFTConfig(
    output_dir="./sft_gemma_output",
    num_train_epochs=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,           # Un poco más alto pero con scheduler
    lr_scheduler_type="linear",   # Baja la intensidad al final
    warmup_steps=5,               # Empieza suave los primeros pasos
    logging_steps=1,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    bf16=torch.cuda.is_available(),
    report_to="none",
    dataset_text_field="text",
    max_length=256,
    packing=False
)

# 4. Inicializar el trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# 5. Entrenar
print("Iniciando entrenamiento con Scheduler para estabilidad...")
trainer.train()
peft_model = trainer.model
print("Entrenamiento finalizado.")

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Iniciando entrenamiento con Scheduler para estabilidad...


Step,Training Loss
1,7.034478
2,7.640939
3,7.034962
4,7.577610
5,7.407941
6,7.449825
7,7.259968
8,7.321291
9,7.639492
10,7.255233


Entrenamiento finalizado.


### Paso 8: Evaluacion Cualitativa Post-SFT (Comparativa Directa)

Ejecutamos exactamente la misma pregunta para verificar como los adaptadores LoRA incorporaron el conocimiento especifico y la precision institucional:


In [12]:
print("=== VALIDACIÓN DE CONOCIMIENTO ESPECÍFICO (POST-SFT) ===\n")

# Nueva pregunta 1: Soporte Técnico
query_1 = "¿Qué tipo de soporte tiene el plan Enterprise y cuánto tardan en responder?"
print(f"Pregunta 1: {query_1}")
print(f"Respuesta: {test_prompt(query_1, peft_model, tokenizer)}")

print("\n" + "-"*50)

# Nueva pregunta 2: Seguridad / API
query_2 = "¿Cómo puedo rotar las llaves de API y en cuánto tiempo expiran las anteriores?"
print(f"Pregunta 2: {query_2}")
print(f"Respuesta: {test_prompt(query_2, peft_model, tokenizer)}")

print("\n" + "-"*50)

# Nueva pregunta 3: Infraestructura
query_3 = "¿En qué regiones opera TechCloud Pro actualmente?"
print(f"Pregunta 3: {query_3}")
print(f"Respuesta: {test_prompt(query_3, peft_model, tokenizer)}")

=== VALIDACIÓN DE CONOCIMIENTO ESPECÍFICO (POST-SFT) ===

Pregunta 1: ¿Qué tipo de soporte tiene el plan Enterprise y cuánto tardan en responder?
Respuesta: El TechCloud Pro ofrece soporte 7/24h mediante tickets o llamada directa en menos de 15 minutos. El TechCloud General tarda 1-2 horas por correo electrónico.

--------------------------------------------------
Pregunta 2: ¿Cómo puedo rotar las llaves de API y en cuánto tiempo expiran las anteriores?
Respuesta: En el portal, ve a Infraestructura > Clusters K8s > Seguridad > Rotar llaves API. Las viejas expiren en 60 segundos.
protoimplRotar las viejas en el momento.

--------------------------------------------------
Pregunta 3: ¿En qué regiones opera TechCloud Pro actualmente?
Respuesta: TechCloud Pro opera en todos los centros de datos HIPAA y SOC2 con cumplimiento anual.


### Paso 9: Guardado de Adaptadores y Fusion con el Modelo Base (`merge_and_unload`)

Guardamos unicamente los pesos de los adaptadores LoRA (ocupan menos de 20 MB). Luego mostramos conceptualmente como fusionar los adaptadores con los pesos base para exportar un modelo monolitico sin latencia adicional:


In [14]:
adapter_path = "./techcloud_gemma_lora"
peft_model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adaptadores LoRA guardados exitosamente en: {adapter_path}")

# Demostracion del concepto de fusion monolitica (merge_and_unload)
print("\nPara produccion o serving en vLLM / Ollama:")
print("model_fused = peft_model.merge_and_unload()")
print("model_fused.save_pretrained(./modelo_final_fusionado)")

Adaptadores LoRA guardados exitosamente en: ./techcloud_gemma_lora

Para produccion o serving en vLLM / Ollama:
model_fused = peft_model.merge_and_unload()
model_fused.save_pretrained(./modelo_final_fusionado)


### Paso 10: Liberacion de Memoria y Recursos

Limpieza de variables y tensores en la GPU:


In [ ]:
del trainer, peft_model, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos de memoria liberados exitosamente.")